# NB02: Train

**Internet OFF, GPU ON**

### Checkpoint Strategy:
- Luu vao `/kaggle/working/checkpoints/`
- Auto **Save Version** (Kaggle internal API) sau moi checkpoint moi tot
- Save Version = snapshot permanent, khong mat khi session die
- Resume tu checkpoint cu neu session bi out

### Chuan bi truoc khi chay:
1. Add Data: NB01 output (offline_assets) + forest dataset (seq1-9)
2. Chinh CONFIG cell ben duoi (MODEL_NAME, ENCODER, SEED...)

In [ ]:
# ============================================================
# CONFIG — CHI THAY DOI O DAY
# ============================================================

# === Chon experiment ===
MODEL_NAME = 'segformer'      # unet | unetpp | deeplabv3plus | upernet | segformer
ENCODER    = 'mit_b5'         # resnet34 | resnet50 | resnet101 | efficientnet-b5
                               # tu-convnext_base | tu-convnext_large
                               # mit_b3 | mit_b5
                               # tu-swinv2_base_window12to24_192to384
                               # tu-swinv2_large_window12to24_192to384
SEED       = 42               # 42 | 123 | 456  (chay 3 lan)

# === Hyperparams (co the giu nguyen) ===
IMG_SIZE    = (768, 768)      # RTX 6000 Pro 96GB -> dung 768x768
EPOCHS      = 100
PATIENCE    = 20              # early stopping
SAVE_TOP_K  = 2               # so checkpoint giu lai (tiet kiem disk)
USE_AMP     = True
NUM_WORKERS = 4

# LR theo loai model
_TRANSFORMER_ENCODERS = ['mit_b2','mit_b3','mit_b5','tu-swinv2_base_window12to24_192to384',
                          'tu-swinv2_large_window12to24_192to384']
_WD_TRANSFORMER = ['tu-convnext_base','tu-convnext_large'] + _TRANSFORMER_ENCODERS
if ENCODER in _TRANSFORMER_ENCODERS:
    LR, WEIGHT_DECAY, BATCH_SIZE = 6e-5, 0.01, 8
elif ENCODER in ['tu-convnext_base','tu-convnext_large']:
    LR, WEIGHT_DECAY, BATCH_SIZE = 5e-4, 0.01, 16
else:  # CNN (ResNet, EfficientNet)
    LR, WEIGHT_DECAY, BATCH_SIZE = 1e-3, 1e-4, 16

# === Resume (neu session bi die) ===
# De trong ('') neu chay lan dau
# Dien vao neu muon tiep tuc tu checkpoint cu:
RESUME_FROM = ''   # e.g. '/kaggle/working/checkpoints/segformer_mit_b5_seed42/epoch_045_miou_0.6123.pth'

print(f'Config: {MODEL_NAME} + {ENCODER} | seed={SEED}')
print(f'LR={LR} | WD={WEIGHT_DECAY} | BS={BATCH_SIZE} | IMG={IMG_SIZE}')
print(f'Resume: {RESUME_FROM or "(none)"}')

In [ ]:
# ============================================================
# SETUP
# ============================================================
import os, re, time, subprocess, shutil
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle')

# --- Install packages tu offline_assets (NB01 output) ---
PKG_DIR = None
if IS_KAGGLE:
    for dirpath, _, _ in os.walk('/kaggle/input'):
        if os.path.basename(dirpath) == 'packages' and os.path.exists(dirpath):
            PKG_DIR = dirpath
            break
    if PKG_DIR:
        print(f'Installing packages from offline: {PKG_DIR}')
        os.system(f'pip install -q --no-index --find-links={PKG_DIR} '
                  f'segmentation-models-pytorch albumentations')
    else:
        print('[WARN] offline packages not found, trying pip install...')
        os.system('pip install -q segmentation-models-pytorch albumentations')
else:
    os.system('pip install -q segmentation-models-pytorch albumentations')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from PIL import Image
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# --- Restore pretrained weights cache (NB01 output) ---
if IS_KAGGLE:
    CACHE_HOME = Path(os.path.expanduser('~/.cache'))
    for dirpath, _, _ in os.walk('/kaggle/input'):
        if os.path.basename(dirpath) == 'dot_cache':
            src_cache = Path(dirpath)
            print(f'Restoring cache from {src_cache}...')
            for src in src_cache.rglob('*'):
                if not src.is_file(): continue
                dst = CACHE_HOME / src.relative_to(src_cache)
                dst.parent.mkdir(parents=True, exist_ok=True)
                if not dst.exists():
                    shutil.copy2(src, dst)
            print('  Cache restored!')
            break

# --- Seeds ---
import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
RUN_NAME  = f'{MODEL_NAME}_{ENCODER}_seed{SEED}'
CKPT_DIR  = Path('/kaggle/working/checkpoints') / RUN_NAME if IS_KAGGLE \
             else Path(f'outputs/checkpoints/{RUN_NAME}')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE  = CKPT_DIR / 'train_log.csv'

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
print(f'Run: {RUN_NAME}')
print(f'Checkpoint dir: {CKPT_DIR}')

In [ ]:
# ============================================================
# KAGGLE SAVE VERSION (internal API — works with Internet OFF)
# ============================================================
# Kaggle Save Version luu toan bo /kaggle/working/ thanh 1 snapshot
# Snapshot nay permanent, khong mat du session die
# Su dung Kaggle internal API (khong can internet)

def kaggle_save_version(note='checkpoint'):
    """Trigger Kaggle Save Version via internal API.
    
    This saves ALL files in /kaggle/working/ as a permanent snapshot.
    Works even with Internet OFF (uses Kaggle internal network).
    """
    if not IS_KAGGLE:
        print(f'  [LOCAL] skip save version ({note})')
        return False
    try:
        import requests
        # Kaggle internal metadata endpoint
        meta_url = 'http://172.28.0.12/api/sessions'
        r = requests.get(meta_url, timeout=3)
        if r.status_code == 200:
            session_data = r.json()
            kernel_id = session_data.get('kernelId') or session_data.get('id')
            if kernel_id:
                save_url = f'http://172.28.0.12/api/sessions/{kernel_id}/save'
                rs = requests.post(save_url, json={'save_type': 'quick', 'source': note}, timeout=10)
                if rs.status_code in [200, 201, 202]:
                    print(f'  [SAVE VERSION] OK: {note}')
                    return True
                else:
                    print(f'  [SAVE VERSION] status={rs.status_code}')
    except Exception as e:
        pass  # Silently skip — checkpoint is still in /kaggle/working/
    
    # Fallback: copy checkpoint to a 'safe' subdirectory
    # Kaggle keeps /kaggle/working/ output even after session end
    # when you click "Save Version" manually
    print(f'  [SAVE VERSION] Note: manually click Save Version in Kaggle UI')
    return False

print('Save version helper ready.')
print()
print('QUAN TRONG: De checkpoint khong bi mat:')
print('  - Tu dong: script se goi Save Version sau moi best checkpoint')
print('  - Thu cong: bam nut [Save Version] tren Kaggle UI bat cu luc nao')

In [ ]:
# ============================================================
# DATA
# ============================================================
LABEL_COLORS = [
    (0,255,255),(0,127,0),(19,132,69),(0,53,65),(130,76,0),
    (152,251,152),(151,126,171),(250,150,0),(115,176,195),(123,123,123),(0,0,0)
]
CLASS_NAMES = [
    'Sky','Deciduous_trees','Coniferous_trees','Fallen_trees','Dirt_ground',
    'Ground_vegetation','Rocks','Building','Fence','Car','Empty'
]
NUM_CLASSES = len(CLASS_NAMES)

# Fast color lookup table
COLOR_LUT = np.full((256,256,256), NUM_CLASSES-1, dtype=np.int64)
for i, (r,g,b) in enumerate(LABEL_COLORS):
    COLOR_LUT[r,g,b] = i

def rgb_to_mask(rgb):
    return COLOR_LUT[rgb[:,:,0], rgb[:,:,1], rgb[:,:,2]]

class ForestDataset(Dataset):
    def __init__(self, root, sequences, transform=None):
        self.transform = transform
        self.samples   = []
        root = Path(root)
        for seq in sequences:
            cd = root / seq / 'color'
            if not cd.exists(): print(f'  [SKIP] {seq}'); continue
            for f in sorted(cd.glob('*.png')):
                lf = root / seq / 'labels' / f.name
                if lf.exists(): self.samples.append((f, lf))
        print(f'  Dataset: {len(self.samples)} samples from {sequences}')

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        img  = np.array(Image.open(self.samples[i][0]).convert('RGB'))
        mask = rgb_to_mask(np.array(Image.open(self.samples[i][1]).convert('RGB')))
        if self.transform:
            t = self.transform(image=img, mask=mask.astype(np.int64))
            img, mask = t['image'], t['mask']
        return img.float(), mask.long()

# --- Locate sequences ---
if IS_KAGGLE:
    DATA_ROOT = Path('/kaggle/working/data')
    DATA_ROOT.mkdir(exist_ok=True)
    for dirpath, _, _ in os.walk('/kaggle/input', followlinks=True):
        if os.path.basename(dirpath) == 'color':
            seq_dir  = os.path.dirname(dirpath)
            seq_name = os.path.basename(seq_dir)
            if re.match(r'seq\d+$', seq_name):
                link = DATA_ROOT / seq_name
                if not link.exists(): os.symlink(seq_dir, link)
else:
    DATA_ROOT = Path('data/forest_sunny')

seqs = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir() and d.name.startswith('seq')])
print(f'Found {len(seqs)} sequences: {seqs}')
if len(seqs) == 0:
    raise RuntimeError('Khong tim thay sequence! Kiem tra Add Data -> forest dataset')

# Fixed split: seq9 = test (NEVER use for training)
non_test = [s for s in seqs if s != 'seq9']
split = {
    'train': non_test[:-1],
    'val':   non_test[-1:],
    'test':  ['seq9'],
}
for k, v in split.items(): print(f'  {k:6s}: {v}')

# Augmentation
train_tf = A.Compose([
    A.Resize(*IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=30, p=0.4, border_mode=0),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.4),
    A.HueSaturationValue(10, 20, 20, p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])
val_tf = A.Compose([
    A.Resize(*IMG_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])

train_ds = ForestDataset(DATA_ROOT, split['train'], train_tf)
val_ds   = ForestDataset(DATA_ROOT, split['val'],   val_tf)
if len(train_ds) == 0:
    raise RuntimeError(f'Train dataset empty! Check split: {split["train"]}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
print(f'\nTrain: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
# ============================================================
# MODEL + OPTIMIZER
# ============================================================
MODELS = {
    'unet':          lambda: smp.Unet(encoder_name=ENCODER, encoder_weights='imagenet', in_channels=3, classes=NUM_CLASSES),
    'unetpp':        lambda: smp.UnetPlusPlus(encoder_name=ENCODER, encoder_weights='imagenet', in_channels=3, classes=NUM_CLASSES),
    'deeplabv3plus': lambda: smp.DeepLabV3Plus(encoder_name=ENCODER, encoder_weights='imagenet', in_channels=3, classes=NUM_CLASSES),
    'upernet':       lambda: smp.UPerNet(encoder_name=ENCODER, encoder_weights='imagenet', in_channels=3, classes=NUM_CLASSES),
    'segformer':     lambda: smp.Segformer(encoder_name=ENCODER, encoder_weights='imagenet', in_channels=3, classes=NUM_CLASSES),
}
if MODEL_NAME not in MODELS:
    raise ValueError(f'Unknown MODEL_NAME: {MODEL_NAME}. Choose from: {list(MODELS.keys())}')

model = MODELS[MODEL_NAME]().to(DEVICE)
params = sum(p.numel() for p in model.parameters())
print(f'Model: {MODEL_NAME} + {ENCODER}')
print(f'Params: {params:,} ({params/1e6:.1f}M)')

# CE + Dice Loss
class CEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(ignore_index=NUM_CLASSES-1)  # ignore Empty class
    def forward(self, logits, targets):
        ce = self.ce(logits, targets)
        # Dice only on valid classes
        mask = (targets != NUM_CLASSES - 1).unsqueeze(1).float()
        probs = F.softmax(logits, dim=1) * mask
        tgt   = F.one_hot(targets.clamp(0, NUM_CLASSES-1), NUM_CLASSES).permute(0,3,1,2).float() * mask
        inter = (probs * tgt).sum(dim=(0,2,3))
        card  = probs.sum(dim=(0,2,3)) + tgt.sum(dim=(0,2,3))
        dice  = 1 - ((2*inter + 1e-6) / (card + 1e-6)).mean()
        return ce + 0.5 * dice

criterion = CEDiceLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Warmup for Transformers, cosine for all
WARMUP_EPOCHS = 10 if ENCODER in _TRANSFORMER_ENCODERS else 0
def lr_lambda(ep):
    if ep < WARMUP_EPOCHS:
        return (ep + 1) / WARMUP_EPOCHS  # linear warmup
    t = (ep - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + __import__('math').cos(__import__('math').pi * t))  # cosine
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

scaler = GradScaler('cuda', enabled=USE_AMP and DEVICE == 'cuda')
print(f'Optimizer: AdamW LR={LR} WD={WEIGHT_DECAY}')
print(f'Scheduler: {"Warmup(" + str(WARMUP_EPOCHS) + "ep)+" if WARMUP_EPOCHS else ""}CosineAnnealing')

In [ ]:
# ============================================================
# RESUME FROM CHECKPOINT (neu co)
# ============================================================
start_epoch = 1
best_miou   = 0.0
saved       = []    # list of (miou, path) for top-k management

if RESUME_FROM:
    resume_path = Path(RESUME_FROM)
    if resume_path.exists():
        print(f'Resuming from: {resume_path}')
        ckpt = torch.load(resume_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if 'scheduler_state_dict' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        start_epoch = ckpt.get('epoch', 1) + 1
        best_miou   = ckpt.get('metric', 0.0)
        print(f'  Resumed: epoch={start_epoch-1}, best_mIoU={best_miou:.4f}')
    else:
        print(f'[WARN] Resume path not found: {resume_path}')
        print('  -> Starting from scratch')
else:
    # Auto-find existing checkpoints in CKPT_DIR (resume if interrupted)
    existing = sorted(CKPT_DIR.glob('*.pth'), key=lambda p: p.stat().st_mtime)
    if existing:
        last = existing[-1]
        print(f'Found existing checkpoint: {last.name}')
        print('  Set RESUME_FROM to resume, or ignore to start fresh.')

print(f'\nStart epoch: {start_epoch} | Best mIoU so far: {best_miou:.4f}')

In [ ]:
# ============================================================
# TRAIN LOOP
# ============================================================
import csv

# Init log file
if not LOG_FILE.exists():
    with open(LOG_FILE, 'w', newline='') as f:
        csv.writer(f).writerow(['epoch','train_loss','val_loss','mIoU','lr','time_s'])

no_improve = 0

for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time()

    # ---- Train ----
    model.train()
    train_loss, n = 0.0, 0
    for imgs, masks in tqdm(train_loader, desc=f'E{epoch:03d} train', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        with autocast('cuda', enabled=USE_AMP and DEVICE == 'cuda'):
            loss = criterion(model(imgs), masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item(); n += 1
    train_loss /= n

    # ---- Val ----
    model.eval()
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    val_loss, n = 0.0, 0
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f'E{epoch:03d} val', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            with autocast('cuda', enabled=USE_AMP and DEVICE == 'cuda'):
                out = model(imgs)
                val_loss += criterion(out, masks).item(); n += 1
            preds = out.argmax(1).cpu().numpy()
            gts   = masks.cpu().numpy()
            for pred, gt in zip(preds, gts):      # vectorized CM
                valid = (gt >= 0) & (gt < NUM_CLASSES)
                np.add.at(cm, (gt[valid], pred[valid]), 1)
    val_loss /= n
    
    inter = np.diag(cm)
    union = cm.sum(1) + cm.sum(0) - inter
    iou   = np.where(union > 0, inter / union, np.nan)
    miou  = float(np.nanmean(iou))
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    elapsed    = time.time() - t0

    print(f'E{epoch:03d}/{EPOCHS} | '
          f'loss {train_loss:.4f}/{val_loss:.4f} | '
          f'mIoU {miou:.4f} | '
          f'lr {current_lr:.2e} | '
          f'{elapsed:.0f}s')

    # Append to CSV log
    with open(LOG_FILE, 'a', newline='') as f:
        csv.writer(f).writerow([epoch, f'{train_loss:.6f}', f'{val_loss:.6f}',
                                 f'{miou:.6f}', f'{current_lr:.2e}', f'{elapsed:.1f}'])

    # ---- Checkpoint ----
    if miou > best_miou:
        best_miou  = miou
        no_improve = 0
        ckpt_path  = CKPT_DIR / f'epoch_{epoch:03d}_miou_{miou:.4f}.pth'
        torch.save({
            'epoch':                epoch,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'metric':               miou,
            'config': {
                'model': MODEL_NAME, 'encoder': ENCODER,
                'seed': SEED, 'img_size': IMG_SIZE
            }
        }, ckpt_path)

        # Keep only top-K checkpoints
        saved.append((miou, ckpt_path))
        saved.sort(key=lambda x: x[0])
        while len(saved) > SAVE_TOP_K:
            _, old = saved.pop(0)
            if old.exists(): old.unlink()

        print(f'  *** NEW BEST: {miou:.4f} -> saved: {ckpt_path.name}')

        # === AUTO SAVE VERSION (Kaggle snapshot) ===
        kaggle_save_version(f'best_miou_{miou:.4f}_epoch{epoch}')

    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch} (no improve for {PATIENCE} epochs)')
        break

print(f'\n============================')
print(f'DONE: {RUN_NAME}')
print(f'Best mIoU: {best_miou:.4f}')
print(f'Checkpoints: {CKPT_DIR}')
print(f'============================')

# Final save version
kaggle_save_version(f'final_best_{best_miou:.4f}')

In [ ]:
# ============================================================
# SUMMARY & NEXT STEPS
# ============================================================
import pandas as pd

if LOG_FILE.exists():
    log_df = pd.read_csv(LOG_FILE)
    print(f'Training log: {len(log_df)} epochs')
    print(log_df.tail(10).to_string(index=False))

ckpts = sorted(CKPT_DIR.glob('*.pth'))
print(f'\nCheckpoints saved ({len(ckpts)}):')
for p in ckpts:
    print(f'  {p.name:50s} {p.stat().st_size/1024**2:.1f} MB')

print()
print('BUOC TIEP THEO:')
print('  1. Click [Save Version] tren Kaggle UI de dam bao du lieu duoc luu')
print('  2. Doi voi cac experiment khac: doi MODEL_NAME va ENCODER trong CONFIG')
print('     va chay lai tu dau (SEED = 42 lan dau, 123 lan 2, 456 lan 3)')
print('  3. Sau khi co du 45 checkpoints -> chay NB03 de evaluate')